# Radish Bank Iris Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/redis/redis-iris-demos/blob/main/notebooks/radish_bank_workshop.ipynb)

Build a simple Radish Bank chatbot with **Context Retriever**, **Agent Memory**, and **LangCache**.

Run cells **top to bottom** in one session (~2 hours). Works locally (Jupyter / VS Code) or on Colab.

1. Setup — install packages, load credentials from `.env`
2. Define Context Surface models and create the surface
3. Seed Context Retriever, Agent Memory, and LangCache
4. Explore each product, then run the chat loop


# Setup

## Install Packages


In [20]:
%pip install -q "context-surfaces>=0.0.1" "httpx>=0.28.0" "openai>=1.68.0" "pydantic>=2.10.0" "python-dotenv>=1.0.1" "redis>=5.2.0" "redisvl[langcache]>=0.16.0"

import sys
print(f"Ready ({sys.executable})")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Ready (/Users/hillary.toh/.pyenv/versions/3.12.5/bin/python)


### Grab workshop files (if Colab)

**Local:** skip the next cell if you already cloned `redis-iris-demos` and opened this notebook from `notebooks/`.

In [21]:
# NBVAL_SKIP
import sys

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/redis/redis-iris-demos.git
    %cd redis-iris-demos
else:
    print("Local: skipped clone — use your existing repo checkout.")

Local: skipped clone — use your existing repo checkout.


## Step 1 — Configure credentials

Create a `.env` file in the project root with your Redis Cloud + OpenAI values, then run the cell below.


In [23]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

DEMO_CUSTOMER_ID = "CUST001"
WORKSHOP_SURFACE_NAME = "radish-bank-workshop"
WORKSHOP_AGENT_NAME = "radish-bank-workshop-agent"
WORKSHOP_MEMORY_NAMESPACE = "radish-bank-workshop"

print("Credentials loaded from .env")

Credentials OK


In [ ]:
import json
import os
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

import httpx
import openai
from redisvl.extensions.cache.llm import LangCacheSemanticCache
from context_surfaces import UnifiedClient, config as cs_config
from context_surfaces.context_model import export_data_model


# ── Settings ──────────────────────────────────────────────────────────────────

def get_settings():
    """Read credentials from os.environ (populated by the .env cell above)."""
    return SimpleNamespace(
        openai_api_key=os.environ.get("OPENAI_API_KEY", ""),
        openai_base_url=os.environ.get("OPENAI_BASE_URL") or None,
        openai_chat_model=os.environ.get("OPENAI_CHAT_MODEL", "gpt-4o-mini"),
        redis_host=os.environ.get("REDIS_HOST", "localhost"),
        redis_port=int(os.environ.get("REDIS_PORT") or 6379),
        redis_username=os.environ.get("REDIS_USERNAME", "default"),
        redis_password=os.environ.get("REDIS_PASSWORD", ""),
        redis_db=int(os.environ.get("REDIS_DB") or 0),
        redis_ssl=os.environ.get("REDIS_SSL", "false").lower() == "true",
        ctx_admin_key=os.environ.get("CTX_ADMIN_KEY", ""),
        mcp_agent_key=os.environ.get("MCP_AGENT_KEY", ""),
        ctx_surface_id=os.environ.get("CTX_SURFACE_ID", ""),
        memory_api_base_url=os.environ.get("MEMORY_API_BASE_URL", ""),
        memory_store_id=os.environ.get("MEMORY_STORE_ID", ""),
        memory_api_key=os.environ.get("MEMORY_API_KEY", ""),
        langcache_host=os.environ.get("LANGCACHE_URL", ""),
        langcache_cache_id=os.environ.get("LANGCACHE_CACHE_ID", ""),
        langcache_api_key=os.environ.get("LANGCACHE_API_KEY", ""),
    )


# ── Context Surface service ────────────────────────────────────────────────────

class ContextSurfaceService:
    """Thin wrapper around the context_surfaces SDK for MCP tool access."""

    def __init__(self, agent_key):
        self._agent_key = agent_key
        self._client = None
        self._tool_cache = None

    def is_configured(self):
        return bool(self._agent_key)

    async def _get_client(self):
        if self._client is None:
            self._client = UnifiedClient()
            await self._client.__aenter__()
        return self._client

    async def list_tools(self):
        if self._tool_cache is None:
            client = await self._get_client()
            tools = await client.list_tools(self._agent_key)
            self._tool_cache = [t if isinstance(t, dict) else t.model_dump() for t in tools]
        return self._tool_cache

    async def call_tool(self, name, args):
        client = await self._get_client()
        result = await client.query_tool(agent_key=self._agent_key, tool_name=name, arguments=args)
        if isinstance(result, dict):
            content = result.get("content", [])
            if content and content[0].get("type") == "text":
                try:
                    return json.loads(content[0]["text"])
                except json.JSONDecodeError:
                    return {"raw_text": content[0]["text"]}
        return result if isinstance(result, dict) else {"result": result}


# ── Memory service ─────────────────────────────────────────────────────────────

class MemoryService:
    """Thin client for the Agent Memory REST API."""

    def __init__(self, base_url, store_id, api_key, namespace="radish-bank-workshop", threshold=0.7):
        self._base = base_url.rstrip("/") if base_url else ""
        self._store = store_id
        key = api_key or ""
        self._key = key if key.lower().startswith(("bearer ", "basic ")) else f"Bearer {key}"
        self._namespace = namespace
        self._threshold = threshold
        self._client = None

    def is_configured(self):
        return bool(self._base and self._store and self._key.strip("Bearer ").strip())

    def _url(self, path):
        return f"{self._base}/v1/stores/{self._store}{path}"

    def _headers(self):
        return {"Authorization": self._key, "Content-Type": "application/json",
                "Accept": "application/json"}

    def _get_client(self):
        if self._client is None:
            self._client = httpx.AsyncClient(timeout=30.0)
        return self._client

    def create_long_term_memory(self, *, text, owner_id, memory_type="semantic",
                                 topics=None, memory_id=None):
        payload = {"memories": [{"id": memory_id or str(uuid.uuid4()), "text": text,
                                  "memoryType": memory_type, "ownerId": owner_id,
                                  "namespace": self._namespace, "topics": topics or []}]}
        with httpx.Client(timeout=30.0) as c:
            c.post(self._url("/long-term-memory"), headers=self._headers(),
                   json=payload).raise_for_status()

    async def asearch_long_term_memory(self, *, text, owner_id, limit=5):
        payload = {"text": text, "similarityThreshold": self._threshold, "filterOp": "all",
                   "limit": limit,
                   "filter": {"ownerId": {"eq": owner_id}, "namespace": {"eq": self._namespace}}}
        resp = await self._get_client().post(self._url("/long-term-memory/search"),
                                              headers=self._headers(), json=payload)
        if resp.status_code == 424:
            return []
        resp.raise_for_status()
        data = resp.json()
        return data.get("items", data.get("memories", []))

    async def add_session_event(self, *, owner_id, session_id, actor_id, role, text,
                                 metadata=None):
        payload = {"actorId": actor_id, "role": role, "content": [{"text": text}],
                   "createdAt": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
                   "metadata": metadata or {}, "sessionId": session_id}
        await self._get_client().post(self._url("/session-memory/events"),
                                       headers=self._headers(), json=payload)

    async def get_session(self, *, owner_id, session_id):
        resp = await self._get_client().get(self._url(f"/session-memory/{session_id}"),
                                             headers=self._headers())
        resp.raise_for_status()
        return resp.json()

    async def delete_session(self, session_id):
        resp = await self._get_client().delete(self._url(f"/session-memory/{session_id}"),
                                                headers=self._headers())
        if resp.status_code not in (200, 204, 404):
            resp.raise_for_status()


# ── Service factory & setup ────────────────────────────────────────────────────

def init_services():
    s = get_settings()
    cs = ContextSurfaceService(s.mcp_agent_key)
    mem = MemoryService(s.memory_api_base_url, s.memory_store_id, s.memory_api_key)
    lc = LangCacheSemanticCache(
        name="radish-bank-workshop",
        server_url=s.langcache_host,
        cache_id=s.langcache_cache_id,
        api_key=s.langcache_api_key,
    )
    return cs, mem, lc, s


async def run_workshop_setup(entities):
    os.environ.update({"CTX_SURFACE_ID": "", "MCP_AGENT_KEY": "",
                        "MEMORY_NAMESPACE": "radish-bank-workshop",
                        "MEMORY_OWNER_ID": DEMO_CUSTOMER_ID})
    settings = get_settings()

    data_model = export_data_model(
        title="radish-bank-workshop",
        description="Radish Bank workshop — Customer, Account, FixedDepositPlan, BankDocument",
        entities=entities,
    )
    api_url = str(cs_config.api_url).rstrip("/")
    headers = {"Content-Type": "application/json", "X-API-Key": settings.ctx_admin_key}

    async with httpx.AsyncClient(timeout=60.0) as client:
        resp = await client.post(
            f"{api_url}/api/v1/context-surfaces", headers=headers,
            json={
                "name": "radish-bank-workshop",
                "description": "Workshop context surface for Radish Bank demo data",
                "data_model": data_model,
                "data_source": {"type": "redis", "connection_config": {
                    "addr": f"{settings.redis_host}:{settings.redis_port}",
                    "username": settings.redis_username or "default",
                    "password": settings.redis_password,
                    "db": settings.redis_db,
                    "tls_enabled": settings.redis_ssl,
                    "pool_size": 10, "min_idle_conns": 2,
                }},
            },
        )
        if resp.status_code != 201:
            raise RuntimeError(f"Failed to create surface ({resp.status_code}): {resp.text}")
        surface_id = str(resp.json()["id"])

        resp = await client.post(
            f"{api_url}/api/v1/context-surfaces/{surface_id}/agent-keys",
            headers=headers, json={"name": "radish-bank-workshop-agent"},
        )
        if resp.status_code != 201:
            raise RuntimeError(f"Failed to create agent key ({resp.status_code}): {resp.text}")
        agent_key = str(resp.json()["key"])

    os.environ.update({"CTX_SURFACE_ID": surface_id, "MCP_AGENT_KEY": agent_key,
                        "MEMORY_NAMESPACE": "radish-bank-workshop",
                        "MEMORY_OWNER_ID": DEMO_CUSTOMER_ID})
    return surface_id, agent_key


async def seed_workshop_context_data(entities, records, *, settings=None):
    settings = settings or get_settings()
    if not settings.ctx_surface_id:
        raise RuntimeError("CTX_SURFACE_ID is missing — run setup first.")
    entity_map = {cls.__name__: cls for cls in entities}
    payloads = dict(records)
    if "BankDocument" not in payloads:
        bd_path = next((p for p in [Path("workshop_data/bank_documents.jsonl"),
                                     Path("notebooks/workshop_data/bank_documents.jsonl")]
                        if p.exists()), Path("workshop_data/bank_documents.jsonl"))
        payloads["BankDocument"] = [json.loads(l) for l in bd_path.read_text().splitlines() if l.strip()]
    imported = {}
    async with UnifiedClient() as client:
        for name, rows in payloads.items():
            result = await client.import_data(
                admin_key=settings.ctx_admin_key,
                context_surface_id=settings.ctx_surface_id,
                records=[entity_map[name](**r) for r in rows],
                on_conflict="overwrite", on_error="fail_fast",
            )
            imported[name] = result.imported
    return imported


# ── Chat agent helpers ─────────────────────────────────────────────────────────

WORKSHOP_SYSTEM_PROMPT = """You are a Radish Bank customer service assistant for demo customer Merv Kwok (customer_id CUST001).

Use MCP tools for balances, accounts, fixed deposit plans, and bank document search.
For filter_* and search_* tools, pass the match as {"value": "..."} — e.g. filter_account_by_customer_id with value CUST001 (never abbreviate to C001).
Fixed deposit plan ids are FD6 and FD12.

When memory context is provided, use it to personalize answers.
Be concise and polite. This is a demo — no long legal disclaimers.
"""


@dataclass
class ChatTurnResult:
    user_message: str
    assistant_message: str
    cache_hit: bool
    tool_calls: list[str] = field(default_factory=list)
    trace: list[str] = field(default_factory=list)


def _event_text(ev):
    """Extract plain text from a memory event's content list."""
    return " ".join(
        i["text"] for i in ev.get("content", [])
        if isinstance(i, dict) and i.get("text")
    ).strip()


async def _check_cache(cache, message, trace):
    """Return a ChatTurnResult on cache hit, else None."""
    hits = await cache.acheck(prompt=message, num_results=1)
    if hits:
        trace.append(f"LangCache HIT (similarity={hits[0].get('similarity', 0):.3f})")
        return ChatTurnResult(message, str(hits[0].get("response", "")), True, trace=trace)
    trace.append("LangCache MISS")
    return None


async def _fetch_memory(memory_service, owner_id, session_id, message, trace):
    """Store the user event then retrieve session history and long-term memories."""
    if not memory_service.is_configured():
        return [], []
    await memory_service.add_session_event(
        owner_id=owner_id, session_id=session_id, actor_id=owner_id,
        role="USER", text=message, metadata={"source": "workshop-notebook"})
    trace.append("Session memory: stored USER event")
    payload = await memory_service.get_session(owner_id=owner_id, session_id=session_id)
    session_events = payload.get("events", []) if isinstance(payload, dict) else []
    trace.append(f"Session memory: {len(session_events)} event(s) in thread")
    memories = await memory_service.asearch_long_term_memory(
        text=message, owner_id=owner_id, limit=5)
    trace.append(f"Long-term memory: {len(memories)} match(es)")
    return session_events, memories


async def _build_prompt(cs_service, message, session_events, memories):
    """Build the OpenAI messages list and tools list from memory context and MCP tools."""
    stm = "\n".join(
        f"{ev.get('role', 'USER').upper()}: {_event_text(ev)}"
        for ev in session_events[-6:] if _event_text(ev)
    )
    ltm = "\n".join(
        f"- {m['text'].strip()}" for m in memories[:5] if m.get("text", "").strip()
    )
    context = "\n\n".join(filter(None, [
        f"Short-term memory:\n{stm}" if stm else "",
        f"Long-term memory:\n{ltm}" if ltm else "",
    ]))
    enriched = message + (f"\n\nRetrieved memory context:\n{context}" if context else "")

    tool_defs = await cs_service.list_tools()
    tool_list = "\n".join(f"- {t['name']}: {t.get('description', '').strip()}" for t in tool_defs)
    system_prompt = WORKSHOP_SYSTEM_PROMPT + f"\n\nAvailable MCP tools:\n{tool_list}\n"

    oai_tools = [
        {"type": "function", "function": {
            "name": t["name"],
            "description": t.get("description", ""),
            "parameters": t.get("inputSchema", {"type": "object", "properties": {}}),
        }}
        for t in tool_defs
    ]

    msgs = [{"role": "system", "content": system_prompt}]
    for ev in (session_events[:-1] if session_events else [])[-8:]:
        text = _event_text(ev)
        if text:
            role = "assistant" if ev.get("role", "").upper() == "ASSISTANT" else "user"
            msgs.append({"role": role, "content": text})
    msgs.append({"role": "user", "content": enriched})
    return msgs, oai_tools


async def _run_llm(oai, model, msgs, oai_tools, cs_service, max_tool_rounds, trace):
    """Run the LLM + tool-call loop; return (final_text, tool_names_called)."""
    tool_names, response = [], None
    for _ in range(max_tool_rounds):
        response = await oai.chat.completions.create(
            model=model, messages=msgs, tools=oai_tools, temperature=0.2)
        msg = response.choices[0].message
        if not msg.tool_calls:
            break
        msgs.append({
            "role": "assistant", "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })
        for tc in msg.tool_calls:
            tool_names.append(tc.function.name)
            trace.append(f"MCP tool call: {tc.function.name}")
            args = {k: v for k, v in json.loads(tc.function.arguments).items() if v is not None}
            result = await cs_service.call_tool(tc.function.name, args)
            msgs.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id})
    final = (response.choices[0].message.content or "").strip() or "(No response.)"
    return final, tool_names


async def _store_turn(memory_service, cache, owner_id, session_id,
                      message, final, store_in_cache, trace):
    """Persist the assistant reply to session memory and optionally to LangCache."""
    if memory_service.is_configured():
        await memory_service.add_session_event(
            owner_id=owner_id, session_id=session_id, actor_id="assistant",
            role="ASSISTANT", text=final, metadata={"source": "workshop-notebook"})
        trace.append("Session memory: stored ASSISTANT event")
    if store_in_cache:
        await cache.astore(prompt=message, response=final)
        trace.append("LangCache stored")


# ── ChatAgent ──────────────────────────────────────────────────────────────────

class ChatAgent:
    """Wraps all services and runs a single conversational turn."""

    def __init__(self, cs_service, memory_service, langcache_service, settings,
                 owner_id=DEMO_CUSTOMER_ID):
        self.cs = cs_service
        self.memory = memory_service
        self.cache = langcache_service
        self.owner_id = owner_id
        self._model = settings.openai_chat_model
        self._oai = openai.AsyncOpenAI(
            api_key=settings.openai_api_key,
            **({"base_url": settings.openai_base_url} if settings.openai_base_url else {}),
        )

    async def turn(self, message, *, session_id, max_tool_rounds=2, store_in_cache=False):
        trace = []
        message = message.strip()

        cached = await _check_cache(self.cache, message, trace)
        if cached:
            return cached

        session_events, memories = await _fetch_memory(
            self.memory, self.owner_id, session_id, message, trace)

        msgs, oai_tools = await _build_prompt(self.cs, message, session_events, memories)

        final, tool_names = await _run_llm(
            self._oai, self._model, msgs, oai_tools, self.cs, max_tool_rounds, trace)

        await _store_turn(self.memory, self.cache, self.owner_id, session_id,
                          message, final, store_in_cache, trace)

        return ChatTurnResult(message, final, False, tool_calls=tool_names, trace=trace)


def print_turn_result(result):
    print(f"User   : {result.user_message}")
    print(f"Agent  : {result.assistant_message}")
    if result.cache_hit:
        print("         [served from LangCache]")
    if result.tool_calls:
        print(f"Tools  : {', '.join(result.tool_calls)}")
    print("Trace  :", " | ".join(result.trace))


## Step 2 — Context Surface data model

A **Context Surface** is a structured view of your Redis data that the LLM can query via auto-generated MCP tools. You define it by writing `ContextModel` classes — one per entity type — where each field carries indexing metadata. The SDK uses those definitions to:

- provision the right Redis index types (tag, text, numeric, vector)
- generate a set of typed MCP tools (`filter_*`, `search_*`, `get_*_by_id`, `find_*_by_*_range`)
- expose those tools to any MCP-compatible LLM client

The result: the LLM can query live Redis data by calling tools, rather than relying on stale training knowledge.

Four entities power this demo — `Customer`, `Account`, `FixedDepositPlan`, and `BankDocument`. Sample data lives in `workshop_data/`.

In [ ]:
from context_surfaces.context_model import ContextField, ContextModel, ContextRelationship


class Customer(ContextModel):

    __redis_key_template__ = "workshop_customer:{customer_id}"

    customer_id: str = ContextField(
        description="Unique customer identifier; demo customer is always CUST001.",
        is_key_component=True,
    )
    name: str = ContextField(description="Customer full name", index="text", weight=2.0)
    segment: str = ContextField(description="Segment e.g. retail", index="tag")
    accounts = ContextRelationship(
        description="Deposit accounts for this customer",
        target="Account",
        source_field="customer_id",
    )


class Account(ContextModel):

    __redis_key_template__ = "workshop_account:{account_id}"

    account_id: str = ContextField(description="Account identifier", is_key_component=True)
    customer_id: str = ContextField(description="Owning customer", index="tag")
    account_type: str = ContextField(description="savings or current", index="tag")
    balance_sgd: float = ContextField(
        description="Balance in SGD", index="numeric", sortable=True
    )
    status: str = ContextField(description="active or inactive", index="tag")
    customer = ContextRelationship(
        description="Account owner",
        target="Customer",
        source_field="customer_id",
    )


class FixedDepositPlan(ContextModel):

    __redis_key_template__ = "workshop_fd_plan:{plan_id}"

    plan_id: str = ContextField(
        description="Primary key: FD6 or FD12 in this demo.",
        is_key_component=True,
    )
    tenure_months: int = ContextField(
        description="Tenure in months", index="numeric", sortable=True
    )
    rate_percent: float = ContextField(
        description="Annual rate percent", index="numeric", sortable=True
    )
    min_deposit_sgd: int = ContextField(
        description="Minimum deposit SGD", index="numeric", sortable=True
    )


class BankDocument(ContextModel):

    __redis_key_template__ = "workshop_document:{document_id}"

    document_id: str = ContextField(description="Stable document id", is_key_component=True)
    title: str = ContextField(description="Document title", index="text", weight=2.0)
    category: str = ContextField(description="fd_faq, branch_guide, etc.", index="tag")
    content: str = ContextField(description="Markdown body", index="text")
    content_embedding: list[float] = ContextField(
        description="Embedding of content for vector search",
        index="vector",
        vector_dim=1536,
        distance_metric="cosine",
    )


WORKSHOP_ENTITIES = [Customer, Account, FixedDepositPlan, BankDocument]
[cls.__name__ for cls in WORKSHOP_ENTITIES]


['Customer', 'Account', 'FixedDepositPlan', 'BankDocument']

## Step 3 — Create Context Surface

This step calls the Context Surfaces API to register your data model and get back two things:

- **surface_id** — identifies your surface in the API
- **agent_key** — a scoped credential the LLM uses to call MCP tools at runtime

Both are written into `os.environ` so the service clients pick them up automatically via `get_settings()`.

In [25]:
surface_id, agent_key = await run_workshop_setup(WORKSHOP_ENTITIES)
cs_service, memory_service, langcache_service, settings = init_services()
agent = ChatAgent(cs_service, memory_service, langcache_service, settings)
SESSION_ID = f"workshop-{uuid.uuid4()}"

print(f"Surface: {surface_id}")
print(f"Session: {SESSION_ID}")
print(f"Customer: {DEMO_CUSTOMER_ID}")


Surface: 75ea1c41-980e-4642-a939-4b1c6e30a361
Session: workshop-29c4e0d2-654e-41ac-9d9d-5f01175588fd
Customer: CUST001


## Part 1 — Context Retriever

**Context Retriever** is the Redis-backed MCP tool server. Once data is loaded, the LLM can call filter, search, and range tools to fetch exactly the records it needs — no full-table scans, no hallucinated balances.

The cell below defines the records to load. `BankDocument` entries are loaded separately from `workshop_data/bank_documents.jsonl` because they include pre-computed embeddings for vector search. Edit any values here to personalise the demo.

In [26]:
CONTEXT_DATA = {
    "Customer": [
        {"customer_id": "CUST001", "name": "Merv Kwok", "segment": "retail"},
    ],
    "Account": [
        {"account_id": "ACC001", "customer_id": "CUST001", "account_type": "savings", "balance_sgd": 42500.0, "status": "active"},
        {"account_id": "ACC002", "customer_id": "CUST001", "account_type": "current", "balance_sgd": 16200.0, "status": "active"},
    ],
    "FixedDepositPlan": [
        {"plan_id": "FD6", "tenure_months": 6, "rate_percent": 2.8, "min_deposit_sgd": 1000},
        {"plan_id": "FD12", "tenure_months": 12, "rate_percent": 3.1, "min_deposit_sgd": 1000},
    ],
}

for entity, rows in CONTEXT_DATA.items():
    print(f"{entity}: {len(rows)} record(s)")
print("BankDocument: loaded from workshop_data/bank_documents.jsonl during import")

Customer: 1 record(s)
Account: 2 record(s)
FixedDepositPlan: 2 record(s)
BankDocument: loaded from workshop_data/bank_documents.jsonl during import


In [27]:
imported = await seed_workshop_context_data(
    WORKSHOP_ENTITIES,
    CONTEXT_DATA,
    settings=settings,
)

print("Imported:")
for entity, count in imported.items():
    print(f"  {entity}: {count}")

Imported:
  Customer: 1
  Account: 2
  FixedDepositPlan: 2
  BankDocument: 1


## Part 2 — Explore Context Retriever

With data loaded, `cs_service.list_tools()` shows every MCP tool auto-generated from your schema. There will be one tool per indexed field — filters for tag fields, text-search for text fields, range queries for numeric fields, and vector similarity for embedding fields.

You can call any tool directly with `cs_service.call_tool(name, args)` before involving an LLM at all. This is useful for verifying your data model and checking what the agent will actually see.

**Tool argument shape:** `filter_*` and `search_*` tools take a single `value` argument. The field being matched is encoded in the tool name — e.g. `filter_account_by_customer_id` with `{"value": "CUST001"}`.

In [28]:
tools = await cs_service.list_tools()
print(f"{len(tools)} tools:")

for tool in tools:
    print(f"  - {tool['name']}")


16 tools:
  - filter_account_by_account_type
  - filter_account_by_customer_id
  - filter_account_by_status
  - filter_bankdocument_by_category
  - filter_customer_by_segment
  - find_account_by_balance_sgd_range
  - find_fixeddepositplan_by_min_deposit_sgd_range
  - find_fixeddepositplan_by_rate_percent_range
  - find_fixeddepositplan_by_tenure_months_range
  - get_account_by_id
  - get_bankdocument_by_id
  - get_customer_by_id
  - get_fixeddepositplan_by_id
  - search_bankdocument_by_content_embedding_similarity
  - search_bankdocument_by_text
  - search_customer_by_text


In [29]:
accounts = await cs_service.call_tool(
    "filter_account_by_customer_id",
    {"value": DEMO_CUSTOMER_ID},
)
accounts


{'count': 2,
 'has_more': False,
 'limit': 10,
 'offset': 0,
 'results': [{'account_id': 'ACC001',
   'account_type': 'savings',
   'balance_sgd': 42500,
   'customer_id': 'CUST001',
   'id': 'workshop_account:ACC001',
   'status': 'active'},
  {'account_id': 'ACC002',
   'account_type': 'current',
   'balance_sgd': 16200,
   'customer_id': 'CUST001',
   'id': 'workshop_account:ACC002',
   'status': 'active'}],
 'total_count': 2}

## Part 3 — Agent Memory

**Agent Memory** gives the chatbot two layers of memory that persist beyond the LLM's context window:

**Short-term (session) memory** records every message in the current conversation thread. At the start of each turn, the last few exchanges are retrieved and prepended to the prompt — so the LLM remembers what was said earlier in the same chat.

**Long-term memory** stores semantic facts about the user that survive across sessions — preferences, past decisions, recurring needs. These are retrieved by embedding similarity at query time, so only relevant facts are injected rather than the full history.

The cells below seed two long-term facts for the demo customer, then run a similarity search to show what the agent would retrieve for a given question.

In [30]:
MEMORY_SEEDS = [
    {
        "memory_id": "workshop-seed-paperless",
        "text": "Prefers paperless statements and online banking",
        "topics": ["banking", "preferences"],
    },
    {
        "memory_id": "workshop-seed-fd-funding",
        "text": "When placing a fixed deposit, use the Savings Account as the funding account",
        "topics": ["fixed_deposit", "preferences"],
    },
]

for seed in MEMORY_SEEDS:
    memory_service.create_long_term_memory(
        text=str(seed["text"]),
        owner_id=DEMO_CUSTOMER_ID,
        memory_type=seed.get("memory_type", "semantic"),
        topics=list(seed.get("topics") or []),
        memory_id=str(seed["memory_id"]),
    )

print(f"Seeded {len(MEMORY_SEEDS)} long-term memories for {DEMO_CUSTOMER_ID}")
for seed in MEMORY_SEEDS:
    print(f"  - [{seed['memory_id']}] {seed['text']}")


Seeded 2 long-term memories for CUST001
  - [workshop-seed-paperless] Prefers paperless statements and online banking
  - [workshop-seed-fd-funding] When placing a fixed deposit, use the Savings Account as the funding account


In [31]:
memories = await memory_service.asearch_long_term_memory(
    text="fixed deposit funding account preferences",
    owner_id=DEMO_CUSTOMER_ID,
    limit=5,
)
for item in memories:
    print(f"- [{item.get('id', '?')}] {item.get('text', item)}")



- [seed-radish-bank-1] When placing a fixed deposit, use the Savings Account as the funding account
- [workshop-seed-fd-funding] When placing a fixed deposit, use the Savings Account as the funding account


## Part 4 — LangCache

**LangCache** is a semantic cache that sits in front of the LLM. Instead of matching exact strings, it embeds each incoming question and searches for similar questions already answered.

- **HIT**: similarity is above the configured threshold — return the cached answer instantly, no LLM call needed
- **MISS**: no close match — pass through to the LLM as normal, then optionally store the response for future hits

This matters for high-traffic deployments: common questions (rates, hours, FAQs) are answered in milliseconds at near-zero cost. The threshold controls the trade-off between cache coverage and answer freshness.

The cells below seed one prompt/response pair, then test three queries — an exact match, a paraphrase, and an unrelated question.

In [32]:
LANGCACHE_PROMPT = "What are your current fixed deposit interest rates?"

LANGCACHE_RESPONSE = """We currently offer two fixed deposit plans:

- **FD6** (6-month term): **2.8% p.a.** — minimum deposit SGD 1,000
- **FD12** (12-month term): **3.1% p.a.** — minimum deposit SGD 1,000

Interest is calculated daily and paid at maturity. Early withdrawal forfeits all accrued interest.
You can open an FD through your account portal or visit any Radish Bank branch."""


In [33]:
await langcache_service.aclear()
await langcache_service.astore(prompt=LANGCACHE_PROMPT, response=LANGCACHE_RESPONSE)
print("LangCache seed: OK")

LangCache seed: OK


Try exact prompt, paraphrase (HIT), unrelated question (MISS).


In [34]:
for question in [
    LANGCACHE_PROMPT,
    "Tell me about your FD interest rates",
    "What are your branch opening hours?",
]:
    hits = await langcache_service.acheck(prompt=question, num_results=1)
    if hits:
        print(f"HIT ({hits[0].get('similarity', 0):.3f}): {question}")
        if question == LANGCACHE_PROMPT:
            print("  (exact seeded prompt)")
    else:
        print(f"MISS: {question}")

HIT (1.000): What are your current fixed deposit interest rates?
  (exact seeded prompt)
HIT (0.906): Tell me about your FD interest rates
MISS: What are your branch opening hours?


## Part 5 — Chat loop

This is the full pipeline, assembled from the three products above. Each call to `chat_turn` runs:

1. **LangCache check** — if a semantically similar question was answered before, return immediately
2. **Session memory write** — store the user message so it's available for context retrieval
3. **Memory retrieval** — fetch the last few session exchanges (short-term) and relevant long-term facts
4. **Prompt enrichment** — append retrieved memory to the user message before sending to the LLM
5. **LLM + MCP tools** — the model calls Context Retriever tools as needed (balances, plans, documents), then produces a final answer
6. **Session memory write** — store the assistant response to close the loop

Try editing the messages or adding new `chat_turn` cells to explore how memory and caching interact.

In [ ]:
turn1 = await agent.turn(
    "Tell me about your fixed deposit interest rates",
    session_id=SESSION_ID,
)
print_turn_result(turn1)

--- trace ---
LangCache HIT (similarity=0.963)
--- assistant ---
We currently offer two fixed deposit plans:

- **FD6** (6-month term): **2.8% p.a.** — minimum deposit SGD 1,000
- **FD12** (12-month term): **3.1% p.a.** — minimum deposit SGD 1,000

Interest is calculated daily and paid at maturity. Early withdrawal forfeits all accrued interest.
You can open an FD through your account portal or visit any Radish Bank branch.


In [ ]:
turn2 = await agent.turn(
    "What is my savings account balance?",
    session_id=SESSION_ID,
)
print_turn_result(turn2)

--- trace ---
LangCache MISS
Session memory: stored USER event
Session memory: 1 event(s) in thread
Long-term memory: 0 match(es)
MCP tool call: filter_account_by_customer_id
Session memory: stored ASSISTANT event
--- assistant ---
Your savings account balance is SGD 42,500.


In [ ]:
turn3 = await agent.turn(
    "I'd like to place SGD 5,000 in a 6-month fixed deposit. What would potentially be my account balance if I did this?",
    session_id=SESSION_ID,
)
print_turn_result(turn3)

--- trace ---
LangCache MISS
Session memory: stored USER event
Session memory: 3 event(s) in thread
Long-term memory: 2 match(es)
MCP tool call: get_fixeddepositplan_by_id
MCP tool call: get_fixeddepositplan_by_id
Session memory: stored ASSISTANT event
--- assistant ---
You can choose between two fixed deposit plans for your SGD 5,000:

1. **FD6**: 6-month tenure at an interest rate of 2.8%.
2. **FD12**: 12-month tenure at an interest rate of 3.1% (not applicable for your 6-month request).

If you place SGD 5,000 in the 6-month fixed deposit (FD6), the interest earned would be approximately:

- Interest = Principal × Rate × Time
- Interest = 5,000 × 0.028 × (6/12) = SGD 70

Your potential account balance after placing the fixed deposit would be:

- New balance = Current balance - Deposit + Interest
- New balance = 42,500 - 5,000 + 70 = SGD 37,570

So, your potential account balance would be **SGD 37,570** after placing the fixed deposit.


### Restart session

Clear **short-term memory** for the current chat thread, then assign a **new** `SESSION_ID`. Later `chat_turn` cells use that variable, so they need a fresh id after the old session is deleted. Long-term memories from Part 3 are unchanged (those are keyed by customer, not session).


In [ ]:
await memory_service.delete_session(SESSION_ID)
SESSION_ID = f"workshop-{uuid.uuid4()}"
print(f"Session restarted: {SESSION_ID}")

Session restarted: workshop-4c58602d-045a-447b-921d-743a97c6d81e
